# Spiral Waveguide Analysis, Part 1: Generate the Layout

Tracking which device instances came from which design cells is straightforward when you generate the layout, but that information is easy to lose by the time measurements arrive. This notebook writes a design manifest alongside the GDS so that every downstream step can look up waveguide type, width, and length from a table rather than re-parsing the geometry.

Propagation loss is one of the most important performance metrics for a silicon photonics process. It tells you how much optical power the waveguide itself absorbs or scatters per unit length, independent of coupling or device effects. A common way to measure it is the **cutback method**: fabricate a set of waveguides with identical cross-sections but increasing lengths, measure the transmitted power through each, and fit the slope of power vs. length. The slope gives you loss in dB per centimetre.

Spiral waveguides are the standard vehicle for cutback measurements in silicon photonics because they pack a long waveguide into a compact area. A 2 cm spiral fits within a few hundred microns of chip area, whereas a 2 cm straight waveguide would not fit on a reticle at all.

This series covers two waveguide types:
- **Rib waveguides**: partially etched silicon, lower propagation loss, wider mode.
- **Ridge waveguides**: fully etched silicon, higher confinement but more sensitive to sidewall roughness and typically higher loss.

For each type we sweep three widths (0.3, 0.5, and 0.8 µm) at six spiral lengths (0 to 25 mm in 5 mm steps). The length-zero device is just the grating coupler loopback with no waveguide, which serves as the reference for extracting the grating coupler insertion loss from the fit intercept.

The layout is built with [kfactory](https://github.com/gdsfactory/kfactory). Because the spiral geometry is complex, the cell definitions live in `layout.py` alongside this notebook. You do not need to understand the layout details to follow the analysis.

> **Before you start**, make sure your credentials are configured in `~/.gdsfactory/gdsfactoryplus.toml`:
> ```toml
> [tool.gdsfactoryplus.api]
> key = "{your-api-key}"
> 
> [tool.gdsfactoryplus.hub]
> host = "https://{organization}.hub.gdsfactory.com"
> ```

## Setup

In [ ]:
import getpass
from pathlib import Path

import gfhub
import pandas as pd
from layout import top

client = gfhub.Client()
user = getpass.getuser()
print(f"Running as user: {user}")

## Generate the layout

`top()` returns a kfactory top cell containing two sub-cells: `rib_loss` and `ridge_loss`. Each sub-cell holds 18 spiral assemblies (3 widths × 6 lengths). The spiral and grating coupler routing is defined in `layout.py`, which you can inspect if you want to see how the geometry is constructed.

Calling `c.write("spirals.gds")` serialises the full hierarchy to a GDS file. The cell references are preserved, so the file size stays compact even with many device instances.

In [ ]:
c = top()
c.write("spirals.gds")
c

## Build the design manifest

The design manifest is a table that maps each device instance to its physical parameters: waveguide type, width, length, and position on the chip. Downstream analysis will read `width_um` and `length_um` directly from this table rather than parsing the GDS again.

kfactory stores the parameters used to construct each cell in `cell.settings`. We walk the instance hierarchy of the `rib_loss` and `ridge_loss` sub-cells with `begin_instances_rec()`, filter on the assembled device cell names, and collect the settings for each instance.

In [ ]:
def generate_design_manifest(c) -> pd.DataFrame:
    """Walk the GDS hierarchy and build a manifest of spiral device instances."""
    rows = []

    for waveguide_type, top_cell_name, target_pattern in [
        ("rib", "rib_loss", "cutback_rib_assembled*"),
        ("ridge", "ridge_loss", "cutback_ridge_assembled*"),
    ]:
        parent = c.kcl[top_cell_name]
        it = parent.begin_instances_rec()
        it.targets = target_pattern
        while not it.at_end():
            cell = c.kcl[it.inst_cell().cell_index()]
            disp = (it.trans() * it.inst_trans()).disp
            rows.append({
                "cell": cell.name,
                "waveguide_type": waveguide_type,
                "width_um": cell.settings["width"],
                "length_um": cell.settings["length"],
                "x": disp.x,
                "y": disp.y,
            })
            it.next()

    return pd.DataFrame(rows)


manifest = generate_design_manifest(c)
manifest.to_csv("spirals_manifest.csv", index=False)
print(f"{len(manifest)} devices total")
manifest.head(10)

## Clean up existing files (optional)

If you want a fresh start, this cell deletes any previously uploaded GDS and manifest files for this project.

In [ ]:
from contextlib import suppress

for name in ("spirals.gds", "spirals_manifest.csv"):
    for f in client.query_files(name=name, tags=["project:tutorial_spirals", user]):
        with suppress(RuntimeError):
            client.delete_file(f["id"])
            print(f"Deleted {f['original_name']} (id={f['id']})")

## Upload the GDS and manifest

In [ ]:
gds_entry = client.add_file("spirals.gds", tags=["project:tutorial_spirals", user])
print(f"GDS uploaded: id={gds_entry['id']}")

manifest_entry = client.add_file("spirals_manifest.csv", tags=["project:tutorial_spirals", user])
print(f"Manifest uploaded: id={manifest_entry['id']}")

## What's next?

The GDS and design manifest are now in DataLab. The next notebook reads the manifest, simulates swept-wavelength transmission spectra for each spiral device, and uploads them with tags encoding the waveguide type, width, and length so that all downstream analysis can group and query by these parameters.